In [1]:
import warnings
from causalnex.structure import StructureModel
warnings.filterwarnings("ignore")  # silence warnings
from causalnex.structure.notears import from_pandas
from causalnex.plots import plot_structure, NODE_STYLE, EDGE_STYLE
from causalnex.network import BayesianNetwork
import pandas as pd
from causalnex.discretiser import Discretiser
import numpy as np
from sklearn.model_selection import train_test_split
from causalnex.structure.notears import from_pandas_lasso
from IPython.display import Image
from causalnex.evaluation import classification_report
from causalnex.evaluation import roc_auc
from causalnex.inference import InferenceEngine
from causalnex.evaluation import classification_report
import copy
import networkx as nx
import turtorial_utils as utils

In [2]:
df_lt = pd.read_csv("/Users/vladasverkelis/Documents/Doktorantūra/Straipsnis_1/Context_influance_to_EU_structural_funds/Data/y/lt_bn_no_context.csv")

In [3]:
df_lt = df_lt.drop(["Unnamed: 0"], axis = 1)

In [4]:
df_lt

,GDPC,VABI,EMP,QOA,QOR,YUA,REST,EBSG,GGFC,LBGD,FDI,PC,ARP,CO2C,PAM,EUPC
0,8985.274630,29.65,73.0,4.400000,4.970000,8.39,16.482,-13.094653,17.451336,-0.493558,6.55,-11.6,19.1,0.000005,19.195046,45.521991
1,10219.116755,29.20,73.0,4.610000,5.210000,13.35,17.824,-11.598256,18.666542,-2.844590,3.61,-9.0,20.9,0.000005,27.187500,70.724103
2,8528.034573,25.02,67.0,4.220000,5.280000,29.58,19.798,-1.755188,21.205413,-8.626258,-0.96,-13.2,20.3,0.000004,28.797468,337.935130
3,8931.519313,26.21,64.3,3.570000,5.270000,35.74,19.639,-2.029481,20.110710,-7.016844,3.01,-28.9,20.5,0.000005,34.838710,281.231582
4,10259.331961,28.35,66.9,3.730000,5.230000,32.59,19.943,-2.706873,18.467399,-5.541350,4.36,-16.2,19.2,0.000005,30.693069,309.501994
5,11120.403233,28.20,68.5,4.230000,5.170000,26.68,21.437,0.613211,17.543879,-2.893765,1.59,-9.7,18.6,0.000005,36.454849,339.126224
6,11776.792571,27.62,69.9,4.320000,5.020000,21.90,22.689,1.457239,16.735544,-2.417272,1.66,-9.0,20.6,0.000005,39.527027,362.332974
7,12395.979849,27.73,71.8,4.190000,4.940000,19.26,23.592,1.930309,16.691874,-1.911556,0.74,-7.2,19.1,0.000004,41.836735,344.608645
8,12861.323296,26.96,73.3,4.120000,4.960000,16.30,25.748,-0.986813,17.168991,-1.093729,2.50,-10.7,22.2,0.000005,34.707904,29.635159
9,13491.939909,26.00,75.2,4.400000,4.900000,14.45,25.612,0.728400,17.029185,-0.460837,2.74,-12.7,21.9,0.000005,32.986111,115.573864


In [5]:
lt = pd.DataFrame()
lt = df_lt.copy()
lt['CO2C_3'] = lt['CO2C'].shift(3)
lt['VABI_2'] = lt['VABI'].shift(2)
lt['QOA_3'] = lt['QOA'].shift(3)
lt['EBSG_3'] = lt['EBSG'].shift(3)
lt['FDI_1'] = lt['FDI'].shift(1)
lt['ARP_3'] = lt['ARP'].shift(3)

In [6]:
lt = lt.dropna()

In [7]:
sm = StructureModel()

In [8]:
sm.add_edges_from([
    ('EUPC', 'PC'),
    ('CO2C_3', 'PC'),
    ('VABI_2', 'EUPC'),
    ('QOA_3', 'EUPC'),
    ('EBSG_3', 'EUPC'),
    ('FDI_1', 'EUPC'),
    ('ARP_3', 'EUPC'),
])

In [9]:
sm.edges

OutEdgeView([('EUPC', 'PC'), ('CO2C_3', 'PC'), ('VABI_2', 'EUPC'), ('QOA_3', 'EUPC'), ('EBSG_3', 'EUPC'), ('FDI_1', 'EUPC'), ('ARP_3', 'EUPC')])

In [10]:
viz = plot_structure(
    sm,
    all_node_attributes=NODE_STYLE.WEAK,
    all_edge_attributes=EDGE_STYLE.WEAK,
)

viz.toggle_physics(False)
viz.show("Graphs/fully_connected_pl_2007_2015.html")

Graphs/fully_connected_pl_2007_2015.html


In [11]:
bn = BayesianNetwork(sm)

In [16]:
discretised_lt = pd.DataFrame(index=lt.index)

for col in lt.columns:
    no_unique = lt[col].nunique()
    
    if no_unique <= 1:
        discretised_lt[col] = 0
    else:
        try:
            discretised_lt[col] = pd.qcut(
                lt[col], 
                q=min(2, no_unique),  
                labels=False,
                duplicates='drop'
            ).astype(int)
        except ValueError:
            discretised_lt[col] = lt[col].rank(method='dense').astype(int) - 1

print("Discretised data:")
print(discretised_lt)

Discretised data:
    GDPC  VABI  EMP  QOA  QOR  YUA  REST  EBSG  GGFC  LBGD  ...  ARP  CO2C  \
3      0     1    0    0    1    1     0     0     1     0  ...    0     0   
4      0     1    0    0    1    1     0     0     1     0  ...    0     0   
5      0     1    0    0    1    1     0     0     1     0  ...    0     1   
6      0     1    0    1    1    1     0     0     0     0  ...    0     0   
7      0     1    0    0    0    1     0     1     0     0  ...    0     0   
8      0     1    0    0    0    0     1     0     1     1  ...    1     0   
9      0     0    0    1    0    0     1     0     0     1  ...    1     0   
10     1     0    1    1    0    0     1     1     0     1  ...    1     1   
11     1     0    1    1    0    0     0     1     0     1  ...    1     1   
12     1     0    1    1    0    0     0     1     0     1  ...    0     1   
13     1     0    1    0    0    1     1     1     1     0  ...    1     1   
14     1     0    1    0    0    0     1     1

In [17]:
discretised_lt = discretised_lt.reset_index(drop=True)
bn.fit_node_states(discretised_lt)
baseline_auc = utils.get_avg_auc_all_info(discretised_lt, bn)
print(f"Baseline AUC: {baseline_auc}")

Processing fold 0 using 7 cores takes 6.194758892059326 seconds
Processing fold 1 using 7 cores takes 5.646266937255859 seconds
Processing fold 2 using 7 cores takes 5.536287069320679 seconds
Processing fold 3 using 7 cores takes 5.521905899047852 seconds
Processing fold 4 using 7 cores takes 5.728971242904663 seconds
Baseline AUC: 0.539236111111111


In [18]:
edges_to_add = [('LV', 'EUPC'), ('LV', 'PC')]
edges_to_remove = [('EUPC', 'PC')]

bn_with_lv = copy.deepcopy(bn)
bn_with_lv.add_node(
    "LV",
    edges_to_add=edges_to_add,
    edges_to_remove=edges_to_remove,
)

In [19]:
viz = utils.plot_pretty_structure(bn_with_lv.structure, edges_to_highlight=edges_to_add)
viz.show("Graphs/node_added_lt_1.html")

Graphs/node_added_lt_1.html


In [20]:
discretised_lt['LV'] = None
lv_states = [0, 1, 2, 3, 4]

proposed_auc = utils.get_avg_auc_lvs(discretised_lt, bn_with_lv, lv_states)
print(f"AUC from adding LV between 'EUPC' and 'PC': {proposed_auc}")

Processing fold 0 using 7 cores takes 7.836307764053345 seconds
Processing fold 1 using 7 cores takes 6.543547868728638 seconds
Processing fold 2 using 7 cores takes 6.363239049911499 seconds
Processing fold 3 using 7 cores takes 6.361944913864136 seconds
Processing fold 4 using 7 cores takes 6.53516697883606 seconds
AUC from adding LV between 'EUPC' and 'PC': 0.592013888888889


In [21]:
edges_to_add = [('LV', 'VABI_2'), ('LV', 'EUPC')]
edges_to_remove = [('VABI_2', 'EUPC')]

bn_with_lv = copy.deepcopy(bn)
bn_with_lv.add_node(
    "LV",
    edges_to_add=edges_to_add,
    edges_to_remove=edges_to_remove,
)

In [22]:
viz = utils.plot_pretty_structure(bn_with_lv.structure, edges_to_highlight=edges_to_add)
viz.show("Graphs/node_added_lt_2.html")

Graphs/node_added_lt_2.html


In [23]:
discretised_lt['LV'] = None
lv_states = [0, 1, 2, 3, 4]

proposed_auc = utils.get_avg_auc_lvs(discretised_lt, bn_with_lv, lv_states)
print(f"AUC from adding LV between 'VABI_2' and 'EUPC': {proposed_auc}")

Processing fold 0 using 7 cores takes 6.5240089893341064 seconds
Processing fold 1 using 7 cores takes 5.866662263870239 seconds
Processing fold 2 using 7 cores takes 5.821609020233154 seconds
Processing fold 3 using 7 cores takes 5.6877899169921875 seconds
Processing fold 4 using 7 cores takes 6.085873365402222 seconds
AUC from adding LV between 'VABI_2' and 'EUPC': 0.6670138888888889


In [24]:
edges_to_add = [('LV', 'QOA_3'), ('LV', 'EUPC')]
edges_to_remove = [('QOA_3', 'EUPC')]

bn_with_lv = copy.deepcopy(bn)
bn_with_lv.add_node(
    "LV",
    edges_to_add=edges_to_add,
    edges_to_remove=edges_to_remove,
)

In [25]:
viz = utils.plot_pretty_structure(bn_with_lv.structure, edges_to_highlight=edges_to_add)
viz.show("Graphs/node_added_lt_3.html")

Graphs/node_added_lt_3.html


In [26]:
discretised_lt['LV'] = None
lv_states = [0, 1, 2, 3, 4]

proposed_auc = utils.get_avg_auc_lvs(discretised_lt, bn_with_lv, lv_states)
print(f"AUC from adding LV between 'QOA_3' and 'EUPC': {proposed_auc}")

Processing fold 0 using 7 cores takes 6.509164094924927 seconds
Processing fold 1 using 7 cores takes 5.8291099071502686 seconds
Processing fold 2 using 7 cores takes 5.797395944595337 seconds
Processing fold 3 using 7 cores takes 5.892474889755249 seconds
Processing fold 4 using 7 cores takes 5.842945098876953 seconds
AUC from adding LV between 'QOA_3' and 'EUPC': 0.5458333333333334


In [27]:
edges_to_add = [('LV', 'EBSG_3'), ('LV', 'EUPC')]
edges_to_remove = [('EBSG_3', 'EUPC')]

bn_with_lv = copy.deepcopy(bn)
bn_with_lv.add_node(
    "LV",
    edges_to_add=edges_to_add,
    edges_to_remove=edges_to_remove,
)

In [28]:
viz = utils.plot_pretty_structure(bn_with_lv.structure, edges_to_highlight=edges_to_add)
viz.show("Graphs/node_added_lt_4.html")

Graphs/node_added_lt_4.html


In [29]:
discretised_lt['LV'] = None
lv_states = [0, 1, 2, 3, 4]

proposed_auc = utils.get_avg_auc_lvs(discretised_lt, bn_with_lv, lv_states)
print(f"AUC from adding LV between 'EBSG_3' and 'EUPC': {proposed_auc}")

Processing fold 0 using 7 cores takes 6.370677947998047 seconds
Processing fold 1 using 7 cores takes 6.112100839614868 seconds
Processing fold 2 using 7 cores takes 6.1646106243133545 seconds
Processing fold 3 using 7 cores takes 5.918271064758301 seconds
Processing fold 4 using 7 cores takes 5.919538974761963 seconds
AUC from adding LV between 'EBSG_3' and 'EUPC': 0.6465277777777778


In [30]:
edges_to_add = [('LV', 'FDI_1'), ('LV', 'EUPC')]
edges_to_remove = [('FDI_1', 'EUPC')]

bn_with_lv = copy.deepcopy(bn)
bn_with_lv.add_node(
    "LV",
    edges_to_add=edges_to_add,
    edges_to_remove=edges_to_remove,
)

In [31]:
viz = utils.plot_pretty_structure(bn_with_lv.structure, edges_to_highlight=edges_to_add)
viz.show("Graphs/node_added_lt_5.html")

Graphs/node_added_lt_5.html


In [32]:
discretised_lt['LV'] = None
lv_states = [0, 1, 2, 3, 4]

proposed_auc = utils.get_avg_auc_lvs(discretised_lt, bn_with_lv, lv_states)
print(f"AUC from adding LV between 'FDI_1' and 'EUPC': {proposed_auc}")

Processing fold 0 using 7 cores takes 6.653856039047241 seconds
Processing fold 1 using 7 cores takes 5.983689069747925 seconds
Processing fold 2 using 7 cores takes 6.081870079040527 seconds
Processing fold 3 using 7 cores takes 6.043254613876343 seconds
Processing fold 4 using 7 cores takes 6.051857948303223 seconds
AUC from adding LV between 'FDI_1' and 'EUPC': 0.6121527777777779


In [33]:
edges_to_add = [('LV', 'ARP_3'), ('LV', 'EUPC')]
edges_to_remove = [('ARP_3', 'EUPC')]

bn_with_lv = copy.deepcopy(bn)
bn_with_lv.add_node(
    "LV",
    edges_to_add=edges_to_add,
    edges_to_remove=edges_to_remove,
)

In [34]:
viz = utils.plot_pretty_structure(bn_with_lv.structure, edges_to_highlight=edges_to_add)
viz.show("Graphs/node_added_lt_5.html")

Graphs/node_added_lt_5.html


In [35]:
discretised_lt['LV'] = None
lv_states = [0, 1, 2, 3, 4]

proposed_auc = utils.get_avg_auc_lvs(discretised_lt, bn_with_lv, lv_states)
print(f"AUC from adding LV between 'ARP_3' and 'EUPC': {proposed_auc}")

Processing fold 0 using 7 cores takes 6.942883014678955 seconds
Processing fold 1 using 7 cores takes 6.253763198852539 seconds
Processing fold 2 using 7 cores takes 6.001307964324951 seconds
Processing fold 3 using 7 cores takes 5.985255718231201 seconds
Processing fold 4 using 7 cores takes 6.033010959625244 seconds
AUC from adding LV between 'ARP_3' and 'EUPC': 0.7006944444444445


In [67]:
import warnings
from causalnex.structure import StructureModel
warnings.filterwarnings("ignore")  # silence warnings
from causalnex.structure.notears import from_pandas
from causalnex.plots import plot_structure, NODE_STYLE, EDGE_STYLE
from causalnex.network import BayesianNetwork
import pandas as pd
from causalnex.discretiser import Discretiser
import numpy as np
from sklearn.model_selection import train_test_split
from causalnex.structure.notears import from_pandas_lasso
from IPython.display import Image
from causalnex.evaluation import classification_report
from causalnex.evaluation import roc_auc
from causalnex.inference import InferenceEngine
from causalnex.evaluation import classification_report
import copy
import networkx as nx
import turtorial_utils as utils

In [68]:
df_lt_1 = pd.read_csv("/Users/vladasverkelis/Documents/Doktorantūra/Straipsnis_1/Context_influance_to_EU_structural_funds/Data/y/lt_bn_no_context.csv")

In [69]:
df_lt_1 = df_lt_1.drop(["Unnamed: 0"], axis = 1)

In [70]:
df_lt_1

,GDPC,VABI,EMP,QOA,QOR,YUA,REST,EBSG,GGFC,LBGD,FDI,PC,ARP,CO2C,PAM,EUPC
0,8985.274630,29.65,73.0,4.400000,4.970000,8.39,16.482,-13.094653,17.451336,-0.493558,6.55,-11.6,19.1,0.000005,19.195046,45.521991
1,10219.116755,29.20,73.0,4.610000,5.210000,13.35,17.824,-11.598256,18.666542,-2.844590,3.61,-9.0,20.9,0.000005,27.187500,70.724103
2,8528.034573,25.02,67.0,4.220000,5.280000,29.58,19.798,-1.755188,21.205413,-8.626258,-0.96,-13.2,20.3,0.000004,28.797468,337.935130
3,8931.519313,26.21,64.3,3.570000,5.270000,35.74,19.639,-2.029481,20.110710,-7.016844,3.01,-28.9,20.5,0.000005,34.838710,281.231582
4,10259.331961,28.35,66.9,3.730000,5.230000,32.59,19.943,-2.706873,18.467399,-5.541350,4.36,-16.2,19.2,0.000005,30.693069,309.501994
5,11120.403233,28.20,68.5,4.230000,5.170000,26.68,21.437,0.613211,17.543879,-2.893765,1.59,-9.7,18.6,0.000005,36.454849,339.126224
6,11776.792571,27.62,69.9,4.320000,5.020000,21.90,22.689,1.457239,16.735544,-2.417272,1.66,-9.0,20.6,0.000005,39.527027,362.332974
7,12395.979849,27.73,71.8,4.190000,4.940000,19.26,23.592,1.930309,16.691874,-1.911556,0.74,-7.2,19.1,0.000004,41.836735,344.608645
8,12861.323296,26.96,73.3,4.120000,4.960000,16.30,25.748,-0.986813,17.168991,-1.093729,2.50,-10.7,22.2,0.000005,34.707904,29.635159
9,13491.939909,26.00,75.2,4.400000,4.900000,14.45,25.612,0.728400,17.029185,-0.460837,2.74,-12.7,21.9,0.000005,32.986111,115.573864


In [71]:
lt_1 = pd.DataFrame()
lt_1 = df_lt_1.copy()
lt_1['EUPC_2'] = lt_1['EUPC'].shift(2)
lt_1['ARP_2'] = lt_1['ARP'].shift(2)
lt_1['FDI_3'] = lt_1['FDI'].shift(3)

In [72]:
lt_1 = lt_1.dropna()

In [73]:
sm1 = StructureModel()

In [74]:
sm1.add_edges_from([
    ('EUPC_2', 'FDI'),
    ('ARP_2', 'FDI'),
    ('ARP_2', 'CO2C'),
    ('FDI_3', 'CO2C')
])

In [75]:
sm1.edges

OutEdgeView([('EUPC_2', 'FDI'), ('ARP_2', 'FDI'), ('ARP_2', 'CO2C'), ('FDI_3', 'CO2C')])

In [76]:
viz = plot_structure(
    sm1,
    all_node_attributes=NODE_STYLE.WEAK,
    all_edge_attributes=EDGE_STYLE.WEAK,
)

viz.toggle_physics(False)
viz.show("Graphs/fully_connected_lt_4.html")

Graphs/fully_connected_lt_4.html


In [77]:
bn1 = BayesianNetwork(sm1)

In [78]:
discretised_lt = pd.DataFrame(index=lt_1.index)

for col in lt_1.columns:
    no_unique = lt_1[col].nunique()
    
    if no_unique <= 1:
        discretised_lt[col] = 0
    else:
        try:
            discretised_lt[col] = pd.qcut(
                lt_1[col], 
                q=min(3, no_unique),  
                labels=False,
                duplicates='drop'
            ).astype(int)
        except ValueError:
            discretised_lt[col] = lt_1[col].rank(method='dense').astype(int) - 1

print("Discretised data:")
print(discretised_lt)

Discretised data:
    GDPC  VABI  EMP  QOA  QOR  YUA  REST  EBSG  GGFC  LBGD  FDI  PC  ARP  \
3      0     1    0    0    2    2     0     0     2     0    1   0    0   
4      0     2    0    0    2    2     0     0     2     0    2   0    0   
5      0     2    0    0    2    2     0     0     2     0    0   1    0   
6      0     2    0    1    2    2     0     1     0     0    0   1    1   
7      0     2    0    0    0    1     0     1     0     1    0   1    0   
8      1     1    1    0    1    1     1     0     1     1    0   0    2   
9      1     1    1    2    0    1     1     1     1     2    1   0    2   
10     1     1    1    2    0    0     2     2     0     2    1   0    2   
11     1     0    2    2    0    0     1     1     0     2    0   1    2   
12     2     0    2    2    0    0     1     2     1     2    2   2    1   
13     2     0    1    1    1    1     2     2     2     0    2   2    1   
14     2     0    2    1    1    0     2     2     1     1    2   2   

In [79]:
discretised_lt = discretised_lt.reset_index(drop=True)
bn1.fit_node_states(discretised_lt)
baseline_auc = utils.get_avg_auc_all_info(discretised_lt, bn1)
print(f"Baseline AUC: {baseline_auc}")

Processing fold 0 using 7 cores takes 7.021398067474365 seconds
Processing fold 1 using 7 cores takes 5.367685794830322 seconds
Processing fold 2 using 7 cores takes 5.4926159381866455 seconds
Processing fold 3 using 7 cores takes 5.37949800491333 seconds
Processing fold 4 using 7 cores takes 5.326723098754883 seconds
Baseline AUC: 0.5061111111111112


In [80]:
edges_to_add = [('LV', 'EUPC_2'), ('LV', 'FDI')]
edges_to_remove = [('EUPC_2', 'FDI')]

bn_with_lv = copy.deepcopy(bn1)
bn_with_lv.add_node(
    "LV",
    edges_to_add=edges_to_add,
    edges_to_remove=edges_to_remove,
)

In [81]:
viz = utils.plot_pretty_structure(bn_with_lv.structure, edges_to_highlight=edges_to_add)
viz.show("Graphs/node_added_pl_3.html")

Graphs/node_added_pl_3.html


In [82]:
discretised_lt['LV'] = None
lv_states = [0, 1, 2, 3, 4]

proposed_auc = utils.get_avg_auc_lvs(discretised_lt, bn_with_lv, lv_states)
print(f"AUC from adding LV between 'EUPC_2' and 'FDI': {proposed_auc}")

Processing fold 0 using 7 cores takes 5.773234128952026 seconds
Processing fold 1 using 7 cores takes 5.461625099182129 seconds
Processing fold 2 using 7 cores takes 5.4100141525268555 seconds
Processing fold 3 using 7 cores takes 5.388048887252808 seconds
Processing fold 4 using 7 cores takes 5.464235067367554 seconds
AUC from adding LV between 'EUPC_2' and 'FDI': 0.4336111111111111
